<a href="https://colab.research.google.com/github/alwhshalkasr267-design/FinalProject/blob/main/FinalProject_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connect drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

Load Data

In [ ]:
import os
import pandas as pd

folder_path = '/content/drive/MyDrive/Obstacles_dataset'
classes_names = os.listdir(folder_path)
print("Dataset Classes: ", classes_names)

In [ ]:
i = 1
for folder in sorted(os.listdir(folder_path)):
    full_folder_path = os.path.join(folder_path, folder)
    if os.path.isdir(full_folder_path):
        num_images = len(os.listdir(full_folder_path))
        print(f"{i} Number of [{folder}] Class : {num_images} images")
        i += 1


In [ ]:
images = []
labels = []

for label_indx, classes_name in enumerate(classes_names):
  class_dir = os.path.join(folder_path, classes_name)
  if os.path.isdir(class_dir):
        print(f"Class [{classes_name}] is done ,  It got label number: {label_indx}")
        for file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, file)
            images.append(img_path)
            labels.append(str(label_indx))
print(f"\nTotal images collected : {len(images)}")

In [ ]:
df = pd.DataFrame({
    "image" : images,
    "label" : labels
})

df = df.sample(frac=1).reset_index(drop=True)  #Mixing up the raws
df.head()

In [ ]:
from PIL import Image
from tqdm import tqdm

bad = []

for p in tqdm(df["image"]):
  try:
    with Image.open(p) as im:
      im.load()
  except:
    bad.append(p)

print("Bad images : ", len(bad))

df = df[~df["image"].isin(bad)].reset_index(drop=True)
print("Remaining images : ", len(df) )

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import gradio as gr
from PIL import Image
import numpy as np

try:
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes_names, yticklabels=classes_names)
    plt.show()
except NameError:
    pass

def predict_image(img):
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    try:
        prediction = model.predict(img_array)
        return classes_names[np.argmax(prediction)]
    except NameError:
        return "Error: Model not found"

demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Text(label="Result"),
    title="Image Classification System"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f379b65707a9200396.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
